# Code for implementing the traditional Huff and the time-aware dynamic Huff model (T-Huff)

### Reference: Liang, Y., Gao, S., Cai, Y., Foutz, N. Z., & Wu, L. (2020). Calibrating the dynamic Huff model for business analysis using location big data. Transactions in GIS. 24(3), 1-23. DOI: 10.1111/tgis.12624. arXiv preprint available at https://arxiv.org/abs/2003.10857.

### Data: SafeGraph (https://shop.safegraph.com/)

<img src="https://geods.geography.wisc.edu/wp-content/uploads/2020/04/tgis12624_prediction.jpg"
     alt="Preiction vs. Actual"
     style="float: left; margin-right: 10px;" />

# import the pyswarms lib

In [18]:
import sys

In [19]:
# Import modules
import numpy as np

# Import PySwarms
#import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
#%load_ext autoreload
#%autoreload 2

# define the Huff model

In [20]:
def HuffModel(mx,alpha,beta,att_field):
    #get the sum of H (dinominator)
    #print(alpha)
    #print(beta)
    cbg_sum = {}
    for i,row in mx.iterrows():
        if mx.loc[i]["census_block_group"] not in cbg_sum.keys():
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] = HuffPro(A,D,alpha,beta)
        else:
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] += HuffPro(A,D,alpha,beta)
    mx["pro"] = 0.0
    mx["h_sum"] = 0.0
    for i,row in mx.iterrows():
        mx.loc[i,"h_sum"] = cbg_sum[mx.loc[i]["census_block_group"]]
        A = mx.loc[i][att_field]
        D = mx.loc[i]["distance"]
        mx.loc[i,"pro"] =  HuffPro(A,D,alpha,beta)/mx.loc[i]["h_sum"]
    return mx
    

In [21]:
def HuffPro(A,D,alpha,beta):
    return pow(A,alpha)/pow(D,beta)

In [22]:
#"Whole Foods Market"
def Corr(mx,alpha,beta,att_field,location_name):
    mx_hf = HuffModel(mx,alpha,beta,att_field)
    mx_hf["pro"] = mx_hf["pro"].astype(float)
    mx_hf["actual_pro"] = mx_hf["actual_pro"].astype(float)
    mx_hf_location = mx_hf[mx_hf["location_name"] == location_name]
    cor = mx_hf_location[["actual_pro","pro"]].corr().values[0][1]
    #cor_total = mx_hf[["actual_pro","pro"]].corr().values[0][1]
    return mx_hf, cor

In [23]:
def THuffModelHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield):
    cbg_sum = {}
    for i,row in mx.iterrows():
        if mx.loc[i]["census_block_group"] not in cbg_sum.keys():
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] = HuffPro(A,D,alpha,beta)
        else:
            A = mx.loc[i][att_field]
            D = mx.loc[i]["distance"]
            cbg_sum[mx.loc[i]["census_block_group"]] += HuffPro(A,D,alpha,beta)
    mx_location = mx[mx["location_name"] == location_name]
    mx_location["pro_total"] = 0.0
    mx_location["h_sum"] = 0.0
    mx_location["pro_hourly_est"] = None
    mx_location["pro_hourly_actual"] = None
    for i,row in mx_location.iterrows():
        mx_location.loc[i,"h_sum"] = cbg_sum[mx_location.loc[i]["census_block_group"]]
        A = mx_location.loc[i][att_field]
        D = mx_location.loc[i]["distance"]
        mx_location.loc[i,"pro_total"] =  HuffPro(A,D,alpha,beta)/mx_location.loc[i]["h_sum"]
        tem_dic_est = {}
        tem_dic_act = {}
        for key in mx_location.loc[i][hourlyfield].keys():
            tem_dic_est[key] = [x*mx_location.loc[i]["pro_total"] for x in mx_location.loc[i][hourlyfield][key]]
            tem_dic_act[key] = [x*mx_location.loc[i]["actual_pro"] for x in mx_location.loc[i][hourlyfield][key]]
        mx_location.at[i,"pro_hourly_est"] = tem_dic_est
        mx_location.at[i,"pro_hourly_actual"] = tem_dic_act
    return mx_location

In [24]:
def CorrHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield,actual_field,est_field):
    mx_hf_location = THuffModelHourlyLocation(mx,alpha,beta,att_field,location_name,hourlyfield)
    #print(mx_hf)
    actual_list = []
    est_list = []
    for i,row in mx_hf_location.iterrows():
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Monday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Tuesday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Wednesday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Thursday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Friday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Saturday'])
        actual_list.extend(mx_hf_location.loc[i][actual_field]['Sunday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Monday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Tuesday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Wednesday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Thursday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Friday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Saturday'])
        est_list.extend(mx_hf_location.loc[i][est_field]['Sunday'])
        #print(actual_list)
        #print(est_list)
        cor = numpy.corrcoef(actual_list,est_list)[0][1]
    return mx_hf_location, cor

# Read the data we processed. More raw data can be found from SafeGraph Inc. 

In [25]:
import pandas as pd
import numpy
infile = "../data/la_wholefoods.csv"
mx = pd.read_csv(infile,dtype={'census_block_group': str})
mx["raw_visit_counts"] = mx["raw_visit_counts"].astype(float)
mx["distance"] = mx["distance"].astype(float)
import ast
mx["popularity_day_pro"] = mx["popularity_day_pro"].apply(ast.literal_eval)
mx["popularity_hourly_pro"] = mx["popularity_hourly_pro"].apply(ast.literal_eval)

# Traditonal Huff model

## can input alpha, beta directly to get the correlation.

In [26]:
alpha_list_ori_detailed = [0.1,0.5,1,2,5]
beta_list = [0.1,0.5,1,2,5]
result = {}
for alpha in alpha_list_ori_detailed:
    for beta in beta_list:
        att_field = "raw_visit_counts"
        location_name = "Whole Foods Market" 
        mx_hf,cor = Corr(mx,alpha,beta,att_field,location_name)
        result[alpha,beta] = cor
        print(alpha, ",",beta,": ",cor)

0.1 , 0.1 :  0.08425880283525153
0.1 , 0.5 :  0.08482923830964151
0.1 , 1 :  0.08540094345880357
0.1 , 2 :  0.08609477374434608
0.1 , 5 :  0.08568810546570421
0.5 , 0.1 :  0.08431489378437948
0.5 , 0.5 :  0.0849497277151888
0.5 , 1 :  0.0856008779787165
0.5 , 2 :  0.08643392034373974
0.5 , 5 :  0.08614579224451199
1 , 0.1 :  0.08425426133554097
1 , 0.5 :  0.08496591043592748
1 , 1 :  0.08571705144722672
1 , 2 :  0.0867420597199658
1 , 5 :  0.08670241733904005
2 , 0.1 :  0.08374615268209351
2 , 0.5 :  0.0845800695068114
2 , 1 :  0.08550662089654612
2 , 2 :  0.08692301932052353
2 , 5 :  0.08769066455002993
5 , 0.1 :  0.08067898341130086
5 , 0.5 :  0.08153362512365327
5 , 1 :  0.08256709421051824
5 , 2 :  0.08448292591932761
5 , 5 :  0.08823245120857458


## optimization to find the alpha, beta with a range (0,2)

In [27]:
def PSOfunction_Huff(x): 
    print(x)
    import timeit
    #start_time = timeit.default_timer()
    alpha = x[:,0]
    beta = x[:,1]
    i = 0
    max_cor = -9999
    att_field = "raw_visit_counts"
    location_name = "Whole Foods Market"
    n_particles = x.shape[0]
    j = []
    for i in range(n_particles):
        mx_hf,cor = Corr(mx,alpha[i],beta[i],att_field,location_name)
        j.append(-cor)
    print(j)
    return np.array(j)

In [28]:
from pyswarms.single.global_best import GlobalBestPSO

#using Whole Foods with 
x_max = 2* np.ones(2)
x_min = 0* np.ones(2)
bounds = (x_min, x_max)
print(bounds)
options = {'c1': 2, 'c2': 2, 'w': 0.9}
#init_pos = pos
optimizer = GlobalBestPSO(n_particles=10, dimensions=2,options=options, bounds=bounds)

#kwargs={"mx": mx}
cost, pos = optimizer.optimize(PSOfunction_Huff, 10)

2026-10-09 16:28:29,405 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 2, 'c2': 2, 'w': 0.9}


(array([0., 0.]), array([2., 2.]))


pyswarms.single.global_best:   0%|          |0/10

[[1.23182789 1.41727598]
 [0.7494264  1.31776649]
 [1.2440154  0.15870716]
 [1.5949672  1.17118217]
 [0.08513286 1.46294684]
 [1.36321369 1.36389242]
 [1.11676074 1.44473338]
 [1.7787776  1.67477268]
 [0.3961535  1.82493762]
 [1.63536365 0.31739156]]


pyswarms.single.global_best:  10%|█         |1/10, best_cost=-0.0866

...

[np.float64(-0.08643620682477121), np.float64(-0.08642375379872959), np.float64(-0.08417528303438912), np.float64(-0.08416006549049558), np.float64(-0.08360039344596817), np.float64(-0.08692682353832205), np.float64(-0.08443826563861098), np.float64(-0.08455226875569306), np.float64(-0.08579970700600224), np.float64(-0.08495437141506071)]
[[0.96434896 0.60887204]
 [1.72003658 0.67526001]
 [1.71487949 0.07346096]
 [0.81278281 1.38328325]
 [0.8878267  0.61941794]
 [1.02477694 0.19940396]
 [0.78682447 0.05563798]
 [1.82189812 0.85582041]
 [0.93870789 1.9916314 ]
 [1.78740576 1.77454656]]


pyswarms.single.global_best:  90%|█████████ |9/10, best_cost=-0.0869

[np.float64(-0.08514505000460375), np.float64(-0.08506527618408515), np.float64(-0.08388123845994983), np.float64(-0.08613040591574514), np.float64(-0.08516359895896984), np.float64(-0.08443422037357495), np.float64(-0.08421606560949658), np.float64(-0.08534032151358606), np.float64(-0.08670594043900764), np.float64(-0.08668818765519389)]
[[0.27093425 1.40528155]
 [1.88175142 1.64779654]
 [0.01626628 0.83515104]
 [1.23010421 0.2832819 ]
 [1.92328987 1.89246679]
 [0.03600412 0.4861315 ]
 [0.70975038 1.33858018]
 [1.75331942 0.68968506]
 [1.12297644 0.0241125 ]
 [1.87058318 1.93867885]]


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.0869
2026-10-09 16:28:45,259 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.08692682353832205, best pos: [1.83597287 1.99261458]


[np.float64(-0.0858737223413732), np.float64(-0.08652025804532185), np.float64(-0.08518144430245059), np.float64(-0.08453255617837858), np.float64(-0.08681091173937991), np.float64(-0.08478338899122373), np.float64(-0.08604678652550422), np.float64(-0.0850762655103345), np.float64(-0.08406931294086974), np.float64(-0.0868674942868634)]


In [29]:
cost,pos

(np.float64(-0.08692682353832205), array([1.83597287, 1.99261458]))

# T-Huff model

## considering the temporal visit changes
<img src="https://geods.geography.wisc.edu/wp-content/uploads/2020/04/tgis12624_HourlyPlot.png"
     alt="Preiction vs. Actual"
     style="float: left; margin-right: 10px;" />

## can input alpha, beta directly to get the correlation.

In [30]:
alpha_list_ori_detailed = [0.1,0.5,1,2,5]
beta_list = [0.1,0.5,1,2,5]
result_dic_ori_detailed_hourly = {}
for alpha in alpha_list_ori_detailed:
    for beta in beta_list:
        att_field = "raw_visit_counts"
        location_name = "Whole Foods Market" 
        mx_hf,cor = CorrHourlyLocation(mx,alpha,beta,att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')
        result_dic_ori_detailed_hourly[alpha,beta] = cor
        print(alpha, ",",beta,": ",cor)

0.1 , 0.1 :  0.4405067092660764
0.1 , 0.5 :  0.4406402613649673
0.1 , 1 :  0.44076455584979524
0.1 , 2 :  0.44087740506905837
0.1 , 5 :  0.44039582573197245
0.5 , 0.1 :  0.44050664206416823
0.5 , 0.5 :  0.44065921128497426
0.5 , 1 :  0.4408069334478135
0.5 , 2 :  0.4409620275706866
0.5 , 5 :  0.4405335595412992
1 , 0.1 :  0.44046739019944187
1 , 0.5 :  0.44064292759134666
1 , 1 :  0.44082015060330154
1 , 2 :  0.44103234216068166
1 , 5 :  0.4406974950444774
2 , 0.1 :  0.44027065536388077
2 , 0.5 :  0.44048477457055285
2 , 1 :  0.4407153558480764
2 , 2 :  0.4410435751354214
2 , 5 :  0.4409762410591505
5 , 0.1 :  0.43913903637876467
5 , 0.5 :  0.43937601101815554
5 , 1 :  0.439657499167544
5 , 2 :  0.4401631559216136
5 , 5 :  0.4410307723926898


## can also just use optimation to get optimal alpha, beta

In [31]:
def PSOfunction_THuff(x): 
    print(x)
    import timeit
    #start_time = timeit.default_timer()
    alpha = x[:,0]
    beta = x[:,1]
    i = 0
    max_cor = -9999
    att_field = "raw_visit_counts"
    location_name = "Whole Foods Market"
    n_particles = x.shape[0]
    j = []
    for i in range(n_particles):
        mx_hf,cor = CorrHourlyLocation(mx,alpha[i],beta[i],att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')
        j.append(-cor)
    print(j)
    return np.array(j)

In [32]:
from pyswarms.single.global_best import GlobalBestPSO

#using Whole Foods with 
x_max = 2* np.ones(2)
x_min = 0* np.ones(2)
bounds = (x_min, x_max)
print(bounds)
options = {'c1': 2, 'c2': 2, 'w': 0.9}
#init_pos = pos
optimizer = GlobalBestPSO(n_particles=10, dimensions=2,options=options, bounds=bounds)

#kwargs={"mx": mx}
cost, pos = optimizer.optimize(PSOfunction_THuff, 10)

2026-10-09 16:30:06,295 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 2, 'c2': 2, 'w': 0.9}


(array([0., 0.]), array([2., 2.]))


pyswarms.single.global_best:   0%|          |0/10

[[1.81932716 1.47977652]
 [1.50493701 0.23294375]
 [1.53565319 1.61503054]
 [1.8482228  0.65586989]
 [1.40311128 1.53331287]
 [0.72197747 0.47376634]
 [0.81714531 0.74642961]
 [0.51299557 0.65003   ]
 [0.01312911 1.97012118]
 [0.44440067 1.87195425]]


pyswarms.single.global_best:  10%|█         |1/10, best_cost=-0.441

...

[np.float64(-0.44059284607360516), np.float64(-0.44089653990835753), np.float64(-0.44097890107753773), np.float64(-0.4408574636055407), np.float64(-0.4404488612632035), np.float64(-0.44070884548475137), np.float64(-0.44093698234152434), np.float64(-0.4406598240848962), np.float64(-0.44039167490787173), np.float64(-0.4405127658224052)]
[[0.94273532 1.60859488]
 [1.63389813 1.46528511]
 [1.35852186 1.55112953]
 [1.71793994 0.06125178]
 [1.39012566 1.94652687]
 [1.76405272 1.3249995 ]
 [1.17530719 1.52103698]
 [0.37539914 0.18144128]
 [1.48470148 1.47922481]
 [1.34318428 0.63303902]]


pyswarms.single.global_best:  90%|█████████ |9/10, best_cost=-0.441

[np.float64(-0.4409683260707632), np.float64(-0.44093000230278256), np.float64(-0.4409658420620024), np.float64(-0.440319380942556), np.float64(-0.44104870210803493), np.float64(-0.4408735940789396), np.float64(-0.440959123241319), np.float64(-0.44054203774587264), np.float64(-0.44094315523679495), np.float64(-0.4406635780265433)]
[[1.26471632 1.42692329]
 [1.78083032 1.7796964 ]
 [1.18407263 1.4844667 ]
 [0.79009297 1.39031501]
 [1.36168639 0.61472039]
 [1.57330801 0.18221948]
 [0.86343217 1.24515717]
 [1.22487523 0.48100192]
 [1.31880073 1.07611464]
 [1.05104624 0.0761647 ]]


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.441
2026-10-09 16:35:50,361 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.4410532187008469, best pos: [1.31613893 1.9888054 ]


[np.float64(-0.4409361623514072), np.float64(-0.4410053590799529), np.float64(-0.44095070008291076), np.float64(-0.4409156168975189), np.float64(-0.44065365668000145), np.float64(-0.44041550155041836), np.float64(-0.44088651412685964), np.float64(-0.4406132916853305), np.float64(-0.4408310158960554), np.float64(-0.4404495040252555)]


In [33]:
cost, pos

(np.float64(-0.4410532187008469), array([1.31613893, 1.9888054 ]))

## output the data for mapping in Huff model

In [34]:
att_field = "raw_visit_counts"
location_name = "Whole Foods Market"
pos = [0.717,0.805]
alpha = pos[0]
beta = pos[1]
mx_hf,cor = Corr(mx,alpha,beta,att_field,location_name)

In [35]:
cor

np.float64(0.0854177921445047)

In [36]:
mx_hf_wholefoods = mx_hf[mx_hf['location_name'] == "Whole Foods Market"]

In [37]:
list1 = mx_hf_wholefoods['actual_pro']
list2 = mx_hf_wholefoods['pro']

In [38]:
numpy.corrcoef(list1,list2)

array([[1.        , 0.08541779],
       [0.08541779, 1.        ]])

## The correlation result for the estimated hourly visit probability using the T-Huff model compared to the actual visit probability observed in SafeGraph POI visits.

In [39]:
att_field = "raw_visit_counts"
location_name = "Whole Foods Market"
pos = [0.787,0.765]
alpha = pos[0]
beta = pos[1]
mx_hf,cor = CorrHourlyLocation(mx,alpha,beta,att_field,location_name,"popularity_hourly_pro",'pro_hourly_actual','pro_hourly_est')

In [40]:
cor

np.float64(0.44074849924184967)